## Imports e Configuração Inicial

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import re
import math
import pickle
import sys
from collections import Counter

# Garantir acesso aos módulos locais se necessário
sys.path.append('../src')

# Configurar dispositivo para PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Ambiente de execução configurado no dispositivo: {device}")

Ambiente de execução configurado no dispositivo: cpu


## Classes do Modelo PyTorch (BiLSTM) e Tokenizador

In [3]:
# 1. Definição do Tokenizador e Vocabulário para o modelo PyTorch
def tokenize(text):
    text = str(text).lower()
    return re.findall(r'\b[a-z]{2,}\b', text)

class Vocabulary:
    def __init__(self, w2i_dict, pad_token="<PAD>", unk_token="<UNK>"):
        self.w2i = w2i_dict
        self.pad_token = pad_token
        self.unk_token = unk_token

    def encode(self, text, max_len=128):
        tokens = tokenize(text)
        ids = [self.w2i.get(token, self.w2i.get(self.unk_token, 1)) for token in tokens[:max_len]]
        if len(ids) < max_len:
            ids += [self.w2i.get(self.pad_token, 0)] * (max_len - len(ids))
        return ids

# 2. Definição da arquitetura PyTorch
class TextBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim=1, n_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=n_layers, 
            bidirectional=True, 
            dropout=dropout if n_layers > 1 else 0.0, 
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        _, (hidden, _) = self.lstm(embedded)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        return self.fc(hidden)

## Função Sigmoide e Avaliação Forward da DNN NumPy

In [5]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def relu(z):
    return np.maximum(0, z)

def numpy_dnn_predict(X, params):
    """Executa a passagem forward da DNN NumPy salva nos parâmetros."""
    cache_A = X.T
    L = len(params) // 2
    
    for l in range(1, L):
        W = params[f'W{l}']
        b = params[f'b{l}']
        Z = np.dot(W, cache_A) + b
        cache_A = relu(Z)
        
    # Última camada (Sigmoid)
    W_last = params[f'W{L}']
    b_last = params[f'b{L}']
    Z_last = np.dot(W_last, cache_A) + b_last
    A_last = sigmoid(Z_last)
    
    return A_last.T # Retorna vetor de probabilidades (n_samples, 1)

## Carregamento dos Ficheiros e Pesos dos Modelos

In [6]:
import sys
import os
import pickle
import torch
import __main__

# 1. Garantir que a pasta 'src' está no caminho de importação do Python
src_path = os.path.abspath('../src')
if src_path not in sys.path:
    sys.path.append(src_path)

# 2. Importar a classe do ficheiro preprocessing.py
from preprocessing import SimpleTFIDF

# 3. TRUQUE DO PICKLE: Registra a classe no __main__ para o pickle reconhecer o arquivo antigo
__main__.SimpleTFIDF = SimpleTFIDF

# --- 1. CARREGAR MODELO 1: DNN NumPy ---
print("A carregar modelo 1 (NumPy DNN)...")
with open('../models/numpy_dnn.pkl', 'rb') as f:
    numpy_artifacts = pickle.load(f)

vectorizer_numpy = numpy_artifacts['vectorizer']
numpy_params = numpy_artifacts['params']
print("Modelo NumPy carregado com sucesso!")

# --- 2. CARREGAR MODELO 2: PyTorch BiLSTM ---
print("A carregar modelo 2 (PyTorch BiLSTM)...")
checkpoint = torch.load('../models/pytorch_bilstm.pth', map_location=device)

vocab_pytorch = Vocabulary(checkpoint['vocab_w2i'])
pytorch_model = TextBiLSTM(
    vocab_size=checkpoint['vocab_size'],
    embedding_dim=checkpoint['embedding_dim'],
    hidden_dim=checkpoint['hidden_dim']
).to(device)

pytorch_model.load_state_dict(checkpoint['model_state_dict'])
pytorch_model.eval()
print("Modelo PyTorch carregado com sucesso!")

A carregar modelo 1 (NumPy DNN)...
Modelo NumPy carregado com sucesso!
A carregar modelo 2 (PyTorch BiLSTM)...
Modelo PyTorch carregado com sucesso!


## Leitura do Dataset do Professor e Execução da Inferência

In [7]:
# Caminho para o dataset de teste fornecido pelo docente
eval_dataset_path = '../data/test_inputs.csv' # Ajustar conforme instrução do docente

print(f"A ler dados de teste de: {eval_dataset_path}")
eval_df = pd.read_csv(eval_dataset_path)

# Garantir que a coluna de texto existe
text_column = 'text' if 'text' in eval_df.columns else eval_df.columns[0]
test_texts = eval_df[text_column].values

print(f"Total de textos para classificar: {len(test_texts)}")

# ---------------------------------------------------------------------
# INFERÊNCIA MODELO 1: DNN NumPy
# ---------------------------------------------------------------------
X_test_numpy = vectorizer_numpy.transform(test_texts)
numpy_probs = numpy_dnn_predict(X_test_numpy, numpy_params)
numpy_preds = (numpy_probs >= 0.5).astype(int).flatten()

# ---------------------------------------------------------------------
# INFERÊNCIA MODELO 2: PyTorch BiLSTM
# ---------------------------------------------------------------------
encoded_texts = [vocab_pytorch.encode(t, max_len=checkpoint['max_len']) for t in test_texts]
tensor_x = torch.tensor(encoded_texts, dtype=torch.long).to(device)

with torch.no_grad():
    logits = pytorch_model(tensor_x).squeeze(-1)
    pytorch_probs = torch.sigmoid(logits).cpu().numpy()
    pytorch_preds = (pytorch_probs >= 0.5).astype(int)

# Salvar resultados compilados num ficheiro CSV de saída
results_df = pd.DataFrame({
    'text': test_texts,
    'pred_numpy_dnn': numpy_preds,
    'prob_numpy_dnn': numpy_probs.flatten(),
    'pred_pytorch_bilstm': pytorch_preds,
    'prob_pytorch_bilstm': pytorch_probs
})

output_csv_path = '../data/submission_results.csv'
results_df.to_csv(output_csv_path, index=False)
print(f"Inferência concluída! Previsões salvas em '{output_csv_path}'")

A ler dados de teste de: ../data/test_inputs.csv
Total de textos para classificar: 100
Inferência concluída! Previsões salvas em '../data/submission_results.csv'


## Verificação Rápida das Previsões

In [9]:
# Exibir as primeiras 20 previsões para verificação visual
print("\n--- AMOSTRA DAS PREVISÕES GERADAS ---")
print(results_df[['pred_numpy_dnn', 'pred_pytorch_bilstm']].head(20))
print("\nContagem de classes - NumPy DNN:", Counter(numpy_preds))
print("Contagem de classes - PyTorch BiLSTM:", Counter(pytorch_preds))


--- AMOSTRA DAS PREVISÕES GERADAS ---
    pred_numpy_dnn  pred_pytorch_bilstm
0                0                    0
1                0                    0
2                0                    0
3                0                    0
4                0                    1
5                0                    1
6                0                    1
7                0                    0
8                0                    1
9                0                    1
10               0                    0
11               0                    1
12               0                    0
13               1                    0
14               0                    0
15               0                    1
16               1                    0
17               1                    1
18               0                    1
19               0                    1

Contagem de classes - NumPy DNN: Counter({np.int64(0): 70, np.int64(1): 30})
Contagem de classes - PyTorch BiLSTM: Count